# Transformer model for when prediction (selected 4-hour context)

This notebook trains the baseline and asymmetric transformer regressors for the time-to-event (“when”) task using only the 4-hour BPA event window. The 4-hour window was selected by the stage-1 XGBoost comparison in `9_predictive_maintenance_xgboost_when.ipynb`; this notebook performs no window-size search.

The event sequence comes from the row-aligned 4-hour “where” dataset. The regression target and fold assignments come from the matching 4-hour “when” dataset.

In [ ]:
import json
import os
import random
from pathlib import Path

# Configure reproducibility before importing/initializing PyTorch.
RANDOM_STATE = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import mean_squared_error, r2_score
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


set_seed()
plt.rcParams.update({"axes.labelsize": 14})

DATA_DIR = Path("output")
OUTPUT_DIR = Path("model_outputs")

WINDOW_SELECTION_NOTEBOOK = "9_predictive_maintenance_xgboost_when.ipynb"
WINDOW_SELECTION_MODEL = "XGBoost"
WINDOW_SELECTION_METADATA_PATH = OUTPUT_DIR / "xgboost_when" / "selected_window.json"
if not WINDOW_SELECTION_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {WINDOW_SELECTION_METADATA_PATH}. Run stage 1 of "
        f"{WINDOW_SELECTION_NOTEBOOK} to select and record the event window first."
    )

try:
    window_selection = json.loads(WINDOW_SELECTION_METADATA_PATH.read_text())
except json.JSONDecodeError as exc:
    raise ValueError(f"Invalid JSON in {WINDOW_SELECTION_METADATA_PATH}.") from exc

required_selection_keys = {
    "selected_window_hours",
    "selection_metric",
    "selection_metric_value",
    "selected_dataset_path",
    "window_metrics_path",
    "selection_source",
}
missing_selection_keys = sorted(required_selection_keys - set(window_selection))
if missing_selection_keys:
    raise ValueError(
        f"Selection metadata is missing required keys: {missing_selection_keys}."
    )

expected_selection_source = "stage_1_xgboost_window_comparison"
expected_selection_metric = "test_mae_seconds"
if window_selection["selection_source"] != expected_selection_source:
    raise ValueError(
        "Unexpected selection source: "
        f"{window_selection['selection_source']!r}; "
        f"expected {expected_selection_source!r}."
    )
if window_selection["selection_metric"] != expected_selection_metric:
    raise ValueError(
        "Unexpected selection metric: "
        f"{window_selection['selection_metric']!r}; "
        f"expected {expected_selection_metric!r}."
    )

selected_window_hours = window_selection["selected_window_hours"]
selection_metric_value = window_selection["selection_metric_value"]
if (
    isinstance(selected_window_hours, bool)
    or not isinstance(selected_window_hours, (int, float))
    or not np.isfinite(selected_window_hours)
    or selected_window_hours <= 0
):
    raise ValueError(f"Invalid selected_window_hours: {selected_window_hours!r}.")
if (
    isinstance(selection_metric_value, bool)
    or not isinstance(selection_metric_value, (int, float))
    or not np.isfinite(selection_metric_value)
):
    raise ValueError(f"Invalid selection_metric_value: {selection_metric_value!r}.")
for path_key in ("selected_dataset_path", "window_metrics_path"):
    path_value = window_selection[path_key]
    if not isinstance(path_value, str) or not path_value.strip():
        raise ValueError(f"Invalid {path_key}: {path_value!r}.")

WINDOW_HOURS = float(selected_window_hours)
if WINDOW_HOURS.is_integer():
    WINDOW_HOURS = int(WINDOW_HOURS)
WINDOW_LABEL = f"{WINDOW_HOURS:g}h"
WINDOW_SELECTION_SOURCE = window_selection["selection_source"]
WINDOW_SELECTION_METRIC = window_selection["selection_metric"]
WINDOW_SELECTION_METRIC_VALUE = float(selection_metric_value)
WINDOW_SELECTION_DATASET_PATH = Path(window_selection["selected_dataset_path"])
WINDOW_SELECTION_SUMMARY_PATH = Path(window_selection["window_metrics_path"])

WHEN_DATASET_PATH = DATA_DIR / f"dataset_winsize{WINDOW_LABEL}_when.csv"
SEQUENCE_DATASET_PATH = DATA_DIR / f"dataset_winsize{WINDOW_LABEL}_where.csv"
if WINDOW_SELECTION_DATASET_PATH.resolve() != WHEN_DATASET_PATH.resolve():
    raise ValueError(
        "Selection metadata dataset does not match selected_window_hours: "
        f"{WINDOW_SELECTION_DATASET_PATH} != {WHEN_DATASET_PATH}."
    )
missing_dataset_paths = [
    path for path in (WHEN_DATASET_PATH, SEQUENCE_DATASET_PATH) if not path.exists()
]
if missing_dataset_paths:
    raise FileNotFoundError(
        f"Missing selected {WINDOW_LABEL} BPA dataset(s): "
        + ", ".join(str(path) for path in missing_dataset_paths)
    )
if not WINDOW_SELECTION_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Selection metadata references a missing window metrics file: "
        f"{WINDOW_SELECTION_SUMMARY_PATH}."
    )

MODEL_OUTPUT_DIR = OUTPUT_DIR / "transformer_when" / f"winsize{WINDOW_LABEL}"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SECONDS_PER_HOUR = 60 * 60
TRAIN_FOLDS = [0, 1, 2, 3, 4]
TEST_FOLD = -1
REGRESSION_TARGET = "label_time_to_event_seconds"
REGRESSION_SUMMARY_FILENAME = f"regression_summary_Transformer_{WINDOW_LABEL}.csv"
MODEL_CONFIGURATION_FILENAME = f"model_configuration_{WINDOW_LABEL}.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using {DEVICE = }")
print(f"Using {RANDOM_STATE = }")
print(
    f"Using selected {WINDOW_HOURS = } from {WINDOW_SELECTION_METADATA_PATH} "
    f"({WINDOW_SELECTION_METRIC}={WINDOW_SELECTION_METRIC_VALUE:.6f})"
)
print(f"Writing {WINDOW_LABEL} artifacts under {MODEL_OUTPUT_DIR}")

## Load and align the selected 4-hour data

The dataset-generation notebook creates the “when” and “where” files from the same ordered samples and assigns the same folds. The checks below prevent training unless the selected 4-hour pair remains row-aligned.

In [ ]:
where_df = pd.read_csv(
    SEQUENCE_DATASET_PATH,
    dtype={"fold_id": "int64"},
    converters={"window": json.loads, "label": json.loads},
)
when_df = pd.read_csv(
    WHEN_DATASET_PATH,
    dtype={"fold_id": "int64"},
)

if len(where_df) != len(when_df):
    raise ValueError(
        "The selected 4-hour sequence and when datasets have different row counts: "
        f"{len(where_df)} != {len(when_df)}"
    )
if not np.array_equal(
    where_df["fold_id"].to_numpy(),
    when_df["fold_id"].to_numpy(),
):
    raise ValueError("Fold IDs are not row-aligned between the 4-hour datasets.")
if not np.array_equal(
    where_df["is_bg"].to_numpy(),
    when_df["is_bg"].to_numpy(),
):
    raise ValueError("Background labels are not row-aligned between the datasets.")

data_df = where_df.copy()
data_df["target_seconds"] = when_df[REGRESSION_TARGET].astype(float)
data_df = data_df.loc[~when_df["is_bg"]].reset_index(drop=True)

if data_df["target_seconds"].isna().any():
    raise ValueError("The filtered 4-hour dataset contains missing time targets.")

print(f"Sequence dataset: {SEQUENCE_DATASET_PATH}")
print(f"When dataset:    {WHEN_DATASET_PATH}")
print(f"Usable samples:   {len(data_df):,}")
display(data_df.groupby("fold_id").size().rename("samples").to_frame())
target_summary = (
    data_df["target_seconds"].agg(["count", "mean", "std", "min", "max"]).to_frame()
)
target_summary.loc["25%"] = data_df["target_seconds"].quantile(0.25)
target_summary.loc["75%"] = data_df["target_seconds"].quantile(0.75)
display(target_summary)

In [ ]:
OUTAGE_TYPE_TO_ID = {"Planned": 0, "Auto": 1}


class WhenOutageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        window = row["window"]
        return {
            "outage_type": torch.tensor(
                [OUTAGE_TYPE_TO_ID[event["outage_type"]] for event in window],
                dtype=torch.long,
            ),
            "from_zone_indices": torch.tensor(
                [event["from_zone_index"] for event in window],
                dtype=torch.long,
            ),
            "to_zone_indices": torch.tensor(
                [event["to_zone_index"] for event in window],
                dtype=torch.long,
            ),
            "time_interval_index": torch.tensor(
                [event["time_interval_index"] for event in window],
                dtype=torch.long,
            ),
            # log1p supports zero-second targets while keeping the model output logarithmic.
            "target_log_seconds": torch.tensor(
                np.log1p(row["target_seconds"]),
                dtype=torch.float32,
            ),
        }


def collate_when_samples(samples):
    lengths = torch.tensor(
        [len(sample["outage_type"]) for sample in samples],
        dtype=torch.long,
    )
    max_length = int(lengths.max())
    padding_mask = torch.arange(max_length).unsqueeze(0) >= lengths.unsqueeze(1)
    return {
        "outage_type": pad_sequence(
            [sample["outage_type"] for sample in samples],
            batch_first=True,
            padding_value=0,
        ),
        "from_zone_indices": pad_sequence(
            [sample["from_zone_indices"] for sample in samples],
            batch_first=True,
            padding_value=0,
        ),
        "to_zone_indices": pad_sequence(
            [sample["to_zone_indices"] for sample in samples],
            batch_first=True,
            padding_value=0,
        ),
        "time_interval_index": pad_sequence(
            [sample["time_interval_index"] for sample in samples],
            batch_first=True,
            padding_value=0,
        ),
        "lengths": lengths,
        "padding_mask": padding_mask,
        "target_log_seconds": torch.stack(
            [sample["target_log_seconds"] for sample in samples]
        ),
    }


def build_window_context(window_df):
    empty_window_mask = window_df["window"].map(len).eq(0)
    if empty_window_mask.any():
        raise ValueError(
            f"Found {int(empty_window_mask.sum())} empty event sequences; "
            "the transformer requires at least one event per sample."
        )

    train_dataset = WhenOutageDataset(window_df[window_df["fold_id"].isin(TRAIN_FOLDS)])
    test_dataset = WhenOutageDataset(window_df[window_df["fold_id"] == TEST_FOLD])
    if not len(train_dataset) or not len(test_dataset):
        raise ValueError("Every window must contain both training and test samples.")

    all_events = [event for window in window_df["window"] for event in window]
    model_num_zones = (
        max(
            max(event["from_zone_index"], event["to_zone_index"])
            for event in all_events
        )
        + 1
    )
    model_num_time_intervals = (
        max(event["time_interval_index"] for event in all_events) + 1
    )
    return train_dataset, test_dataset, model_num_zones, model_num_time_intervals


train_when_dataset, test_when_dataset, num_zones, num_time_intervals = (
    build_window_context(data_df)
)
num_outage_types = len(OUTAGE_TYPE_TO_ID)

print(f"{len(train_when_dataset) = :,}")
print(f"{len(test_when_dataset) = :,}")
print(f"{num_zones = }")
print(f"{num_time_intervals = }")

## Transformer definition

The model follows the event-embedding and transformer-encoder design in the “where” notebook. The only task-specific change is the scalar regression head. Mini-batches use a padding mask, and the representation of the last real event is used for prediction.

In [ ]:
MODEL_SCALE = 1

ZONE_EMBEDDING_DIM = int(64 * MODEL_SCALE)
OUTAGE_TYPE_EMBEDDING_DIM = int(32 * MODEL_SCALE)
TIME_INTERVAL_EMBEDDING_DIM = int(32 * MODEL_SCALE)
TRANSFORMER_DIM = int(256 * MODEL_SCALE)

DROPOUT = 0.15
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 64
BATCH_SIZE = 32

In [ ]:
class WhenTransformer(nn.Module):
    """Transformer regressor over the preceding outage-event sequence."""

    def __init__(
        self,
        num_zones,
        num_outage_types,
        num_time_intervals,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        time_interval_embedding_dim=TIME_INTERVAL_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types, outage_type_embedding_dim
        )
        self.time_interval_embedding = nn.Embedding(
            num_time_intervals, time_interval_embedding_dim
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim
            + outage_type_embedding_dim
            + time_interval_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.projection = nn.Linear(transformer_dim, 1)

    def forward(
        self,
        from_zone_indices,
        to_zone_indices,
        outage_type,
        time_interval_index,
        lengths,
        padding_mask,
    ):
        x = torch.cat(
            [
                self.zone_embedding(from_zone_indices),
                self.zone_embedding(to_zone_indices),
                self.outage_type_embedding(outage_type),
                self.time_interval_embedding(time_interval_index),
            ],
            dim=-1,
        )
        x = self.input_fc(x)
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        last_event_index = lengths - 1
        pooled = x[
            torch.arange(x.shape[0], device=x.device),
            last_event_index,
        ]
        return self.projection(pooled).squeeze(-1)

## Leading and lagging penalties

This standalone settings cell controls the asymmetric objective. A leading prediction estimates an event sooner than it occurs (prediction < target); a lagging prediction estimates it later than it occurs (prediction > target).

In [ ]:
LEADING_PREDICTION_PENALTY = 1.0
LAGGING_PREDICTION_PENALTY = 5.0

In [ ]:
def make_data_loaders(train_dataset, test_dataset, seed=RANDOM_STATE):
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_when_samples,
        generator=generator,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_when_samples,
    )
    return train_loader, test_loader


def batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def predict_batch(model, batch):
    return model(
        batch["from_zone_indices"],
        batch["to_zone_indices"],
        batch["outage_type"],
        batch["time_interval_index"],
        batch["lengths"],
        batch["padding_mask"],
    )


def mae_loss_log_seconds(prediction, target):
    return torch.abs(prediction - target).mean()


def asymmetric_mae_loss_log_seconds(prediction, target):
    residual = prediction - target
    weights = torch.where(
        residual < 0,
        torch.as_tensor(
            LEADING_PREDICTION_PENALTY,
            dtype=prediction.dtype,
            device=prediction.device,
        ),
        torch.as_tensor(
            LAGGING_PREDICTION_PENALTY,
            dtype=prediction.dtype,
            device=prediction.device,
        ),
    )
    return (weights * residual.abs()).mean()


def run_epoch(model, data_loader, loss_fn, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    objective_sum = 0.0
    absolute_error_sum = 0.0
    sample_count = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for batch in data_loader:
            batch = batch_to_device(batch, DEVICE)
            if is_training:
                optimizer.zero_grad(set_to_none=True)

            prediction = predict_batch(model, batch)
            target = batch["target_log_seconds"]
            loss = loss_fn(prediction, target)

            if is_training:
                loss.backward()
                optimizer.step()

            batch_size = target.shape[0]
            objective_sum += loss.item() * batch_size
            prediction_seconds = torch.expm1(prediction)
            target_seconds = torch.expm1(target)
            absolute_error_sum += (
                torch.abs(prediction_seconds - target_seconds).sum().item()
            )
            sample_count += batch_size

    mae_seconds = absolute_error_sum / sample_count
    return {
        "objective": objective_sum / sample_count,
        "mae_seconds": mae_seconds,
    }


def train_when_model(
    loss_fn,
    model_label,
    train_dataset,
    test_dataset,
    model_num_zones,
    model_num_time_intervals,
    num_epochs=NUM_EPOCHS,
    verbose=True,
):
    set_seed()
    model = WhenTransformer(
        num_zones=model_num_zones,
        num_outage_types=num_outage_types,
        num_time_intervals=model_num_time_intervals,
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    train_loader, test_loader = make_data_loaders(train_dataset, test_dataset)
    history = []

    for epoch in range(1, num_epochs + 1):
        train_metrics = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        test_metrics = run_epoch(model, test_loader, loss_fn)
        history.append(
            {
                "epoch": epoch,
                "train_objective": train_metrics["objective"],
                "test_objective": test_metrics["objective"],
                "train_mae_seconds": train_metrics["mae_seconds"],
                "test_mae_seconds": test_metrics["mae_seconds"],
            }
        )
        if verbose:
            print(
                f"{model_label} | epoch {epoch:03d} | "
                f"train MAE={train_metrics['mae_seconds']:.3f}s | "
                f"test MAE={test_metrics['mae_seconds']:.3f}s | "
                f"train objective={train_metrics['objective']:.6f} | "
                f"test objective={test_metrics['objective']:.6f}"
            )

    return model, pd.DataFrame(history)


def collect_predictions(model, dataset):
    data_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_when_samples,
    )
    true_log_seconds = []
    predicted_log_seconds = []

    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            batch = batch_to_device(batch, DEVICE)
            prediction = predict_batch(model, batch)
            true_log_seconds.append(batch["target_log_seconds"].cpu().numpy())
            predicted_log_seconds.append(prediction.cpu().numpy())

    y_true_seconds = np.expm1(np.concatenate(true_log_seconds))
    y_pred_seconds = np.expm1(np.concatenate(predicted_log_seconds))
    signed_error = y_true_seconds - y_pred_seconds
    absolute_error = np.abs(signed_error)
    leading_mask = signed_error > 0
    lagging_mask = signed_error < 0
    asymmetric_weighted_error = np.where(
        leading_mask,
        LEADING_PREDICTION_PENALTY * absolute_error,
        np.where(
            lagging_mask,
            LAGGING_PREDICTION_PENALTY * absolute_error,
            0.0,
        ),
    )
    return pd.DataFrame(
        {
            "y_true": y_true_seconds,
            "y_pred": y_pred_seconds,
            "signed_error": signed_error,
            "absolute_error": absolute_error,
            "asymmetric_weighted_absolute_error": asymmetric_weighted_error,
            "is_leading_prediction": leading_mask,
            "is_lagging_prediction": lagging_mask,
        }
    )


def regression_metrics(prediction_df):
    absolute_errors = prediction_df["absolute_error"]
    return {
        "mae": float(absolute_errors.mean()),
        "rmse": float(
            mean_squared_error(prediction_df["y_true"], prediction_df["y_pred"]) ** 0.5
        ),
        "r2": float(r2_score(prediction_df["y_true"], prediction_df["y_pred"])),
        "asymmetric_penalized_mae": float(
            prediction_df["asymmetric_weighted_absolute_error"].mean()
        ),
        "leading_prediction_rate": float(prediction_df["is_leading_prediction"].mean()),
        "lagging_prediction_rate": float(prediction_df["is_lagging_prediction"].mean()),
        "leading_prediction_penalty": LEADING_PREDICTION_PENALTY,
        "lagging_prediction_penalty": LAGGING_PREDICTION_PENALTY,
    }


def save_checkpoint(
    model,
    training_history,
    model_name,
    loss_name,
    window_hours,
    model_num_zones,
    model_num_time_intervals,
):
    checkpoint_path = MODEL_OUTPUT_DIR / f"{model_name}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": model_name,
            "loss_name": loss_name,
            "target_transform": "log1p_seconds",
            "window_hours": window_hours,
            "num_zones": model_num_zones,
            "num_outage_types": num_outage_types,
            "num_time_intervals": model_num_time_intervals,
            "zone_embedding_dim": ZONE_EMBEDDING_DIM,
            "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
            "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
            "transformer_dim": TRANSFORMER_DIM,
            "dropout": DROPOUT,
            "outage_type_to_id": OUTAGE_TYPE_TO_ID,
            "training_history": training_history.to_dict(orient="records"),
            "leading_prediction_penalty": LEADING_PREDICTION_PENALTY,
            "lagging_prediction_penalty": LAGGING_PREDICTION_PENALTY,
        },
        checkpoint_path,
    )
    print(f"Saved checkpoint to {checkpoint_path}")
    return checkpoint_path

## Baseline MAE model

In [ ]:
baseline_model, baseline_history = train_when_model(
    mae_loss_log_seconds,
    model_label="Baseline MAE (4h)",
    train_dataset=train_when_dataset,
    test_dataset=test_when_dataset,
    model_num_zones=num_zones,
    model_num_time_intervals=num_time_intervals,
)
baseline_predictions = collect_predictions(baseline_model, test_when_dataset)
baseline_metrics = regression_metrics(baseline_predictions)
baseline_metrics_report = {"window_hours": WINDOW_HOURS, **baseline_metrics}

baseline_checkpoint_path = save_checkpoint(
    baseline_model,
    baseline_history,
    model_name=f"when_transformer_mae_{WINDOW_LABEL}",
    loss_name="mae_log_seconds",
    window_hours=WINDOW_HOURS,
    model_num_zones=num_zones,
    model_num_time_intervals=num_time_intervals,
)
baseline_history.to_csv(
    MODEL_OUTPUT_DIR / f"baseline_training_history_{WINDOW_LABEL}.csv",
    index=False,
)
baseline_predictions.to_csv(
    MODEL_OUTPUT_DIR / f"baseline_test_predictions_{WINDOW_LABEL}.csv",
    index=False,
)
(MODEL_OUTPUT_DIR / f"baseline_metrics_{WINDOW_LABEL}.json").write_text(
    json.dumps(baseline_metrics_report, indent=4)
)

baseline_summary = pd.DataFrame([{"model": "Baseline MAE", **baseline_metrics_report}])
display(baseline_summary)

print(f"Test MAE: {baseline_metrics['mae']:.6f} seconds")
print(f"Test RMSE: {baseline_metrics['rmse']:.6f} seconds")
print(f"Test R2: {baseline_metrics['r2']:.6f}")
print("Leading prediction rate: " f"{baseline_metrics['leading_prediction_rate']:.6%}")
print("Lagging prediction rate: " f"{baseline_metrics['lagging_prediction_rate']:.6%}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    baseline_history["epoch"],
    baseline_history["train_mae_seconds"],
    label="Train",
)
plt.plot(
    baseline_history["epoch"],
    baseline_history["test_mae_seconds"],
    label="Test",
)
plt.xlabel("Epoch")
plt.ylabel("MAE (seconds)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_OUTPUT_DIR / f"baseline_training_mae_{WINDOW_LABEL}.pdf", dpi=300)
plt.show()

absolute_errors = baseline_predictions["absolute_error"]
absolute_errors_hours = absolute_errors / SECONDS_PER_HOUR

plt.figure(figsize=(10, 4))
plt.hist(absolute_errors_hours, bins=50, edgecolor="black")
plt.xlabel("Absolute error (hours)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR / f"transformer_absolute_error_distribution_{WINDOW_LABEL}.pdf",
    dpi=300,
)
plt.show()

signed_errors_hours = baseline_predictions["signed_error"] / SECONDS_PER_HOUR
leading_errors_hours = signed_errors_hours[signed_errors_hours > 0]
lagging_errors_hours = signed_errors_hours[signed_errors_hours < 0]
signed_error_bins_hours = np.linspace(
    signed_errors_hours.min(), signed_errors_hours.max(), 50
)

plt.figure(figsize=(10, 4))
plt.hist(
    lagging_errors_hours,
    bins=signed_error_bins_hours,
    alpha=0.7,
    label="Lagging/late: y - y_hat < 0",
)
plt.hist(
    leading_errors_hours,
    bins=signed_error_bins_hours,
    alpha=0.7,
    label="Leading/early: y - y_hat > 0",
)
plt.axvline(0, color="black", linestyle="--", label="No error")
plt.xlabel("Signed error: y - y_hat (hours)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR / f"transformer_signed_error_distribution_{WINDOW_LABEL}.pdf",
    dpi=300,
)
plt.show()

## Asymmetric leading/lagging-penalty model

This model uses the penalty values from the dedicated settings cell above. It is trained from a fresh, reproducibly seeded initialization.

In [ ]:
asymmetric_model, asymmetric_history = train_when_model(
    asymmetric_mae_loss_log_seconds,
    model_label="Asymmetric MAE (4h)",
    train_dataset=train_when_dataset,
    test_dataset=test_when_dataset,
    model_num_zones=num_zones,
    model_num_time_intervals=num_time_intervals,
)
asymmetric_predictions = collect_predictions(asymmetric_model, test_when_dataset)
asymmetric_metrics = regression_metrics(asymmetric_predictions)
asymmetric_metrics_report = {"window_hours": WINDOW_HOURS, **asymmetric_metrics}

asymmetric_checkpoint_path = save_checkpoint(
    asymmetric_model,
    asymmetric_history,
    model_name=f"when_transformer_asymmetric_mae_{WINDOW_LABEL}",
    loss_name="asymmetric_mae_log_seconds",
    window_hours=WINDOW_HOURS,
    model_num_zones=num_zones,
    model_num_time_intervals=num_time_intervals,
)
asymmetric_history.to_csv(
    MODEL_OUTPUT_DIR / f"asymmetric_training_history_{WINDOW_LABEL}.csv",
    index=False,
)
asymmetric_predictions.to_csv(
    MODEL_OUTPUT_DIR / f"asymmetric_test_predictions_{WINDOW_LABEL}.csv",
    index=False,
)
(MODEL_OUTPUT_DIR / f"asymmetric_metrics_{WINDOW_LABEL}.json").write_text(
    json.dumps(asymmetric_metrics_report, indent=4)
)

asymmetric_summary = pd.DataFrame(
    [{"model": "Asymmetric MAE", **asymmetric_metrics_report}]
)
display(asymmetric_summary)

print(f"Test MAE: {asymmetric_metrics['mae']:.6f} seconds")
print(
    "Asymmetric-penalized MAE "
    f"(leading x{LEADING_PREDICTION_PENALTY:g}, "
    f"lagging x{LAGGING_PREDICTION_PENALTY:g}): "
    f"{asymmetric_metrics['asymmetric_penalized_mae']:.6f} seconds"
)
print(
    "Leading prediction rate: " f"{asymmetric_metrics['leading_prediction_rate']:.6%}"
)
print(
    "Lagging prediction rate: " f"{asymmetric_metrics['lagging_prediction_rate']:.6%}"
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    asymmetric_history["epoch"],
    asymmetric_history["train_mae_seconds"],
    label="Train",
)
plt.plot(
    asymmetric_history["epoch"],
    asymmetric_history["test_mae_seconds"],
    label="Test",
)
plt.xlabel("Epoch")
plt.ylabel("MAE (seconds)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_OUTPUT_DIR / f"asymmetric_training_mae_{WINDOW_LABEL}.pdf", dpi=300)
plt.show()

weighted_signed_errors_hours = asymmetric_predictions["signed_error"] / SECONDS_PER_HOUR
weighted_leading_errors_hours = weighted_signed_errors_hours[
    weighted_signed_errors_hours > 0
]
weighted_lagging_errors_hours = weighted_signed_errors_hours[
    weighted_signed_errors_hours < 0
]
weighted_signed_error_bins_hours = np.linspace(
    weighted_signed_errors_hours.min(),
    weighted_signed_errors_hours.max(),
    50,
)

plt.figure(figsize=(10, 4))
plt.hist(
    weighted_lagging_errors_hours,
    bins=weighted_signed_error_bins_hours,
    alpha=0.7,
    label="Lagging/late: y - y_hat < 0",
)
plt.hist(
    weighted_leading_errors_hours,
    bins=weighted_signed_error_bins_hours,
    alpha=0.7,
    label="Leading/early: y - y_hat > 0",
)
plt.axvline(0, color="black", linestyle="--", label="No error")
plt.xlabel("Signed error: y - y_hat (hours)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"asymmetric_transformer_signed_error_distribution_{WINDOW_LABEL}.pdf",
    dpi=300,
)
plt.show()

## Comparison report

In [ ]:
comparison_summary = pd.DataFrame(
    [
        {"model": "Baseline MAE", **baseline_metrics_report},
        {"model": "Asymmetric MAE", **asymmetric_metrics_report},
    ]
)
comparison_summary.to_csv(
    MODEL_OUTPUT_DIR / REGRESSION_SUMMARY_FILENAME,
    index=False,
)

model_configuration = {
    "window_hours": WINDOW_HOURS,
    "window_selected_externally": True,
    "window_selection_notebook": WINDOW_SELECTION_NOTEBOOK,
    "window_selection_model": WINDOW_SELECTION_MODEL,
    "window_selection_metadata_path": str(WINDOW_SELECTION_METADATA_PATH),
    "window_selection_source": WINDOW_SELECTION_SOURCE,
    "window_selection_metric": WINDOW_SELECTION_METRIC,
    "window_selection_metric_value": WINDOW_SELECTION_METRIC_VALUE,
    "window_selection_dataset_path": str(WINDOW_SELECTION_DATASET_PATH),
    "window_selection_summary_path": str(WINDOW_SELECTION_SUMMARY_PATH),
    "window_search_performed": False,
    "target_transform": "log1p_seconds",
    "when_dataset_path": str(WHEN_DATASET_PATH),
    "sequence_dataset_path": str(SEQUENCE_DATASET_PATH),
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "train_folds": TRAIN_FOLDS,
    "test_fold": TEST_FOLD,
    "random_state": RANDOM_STATE,
    "device": str(DEVICE),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "zone_embedding_dim": ZONE_EMBEDDING_DIM,
    "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
    "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
    "transformer_dim": TRANSFORMER_DIM,
    "dropout": DROPOUT,
    "leading_prediction_penalty": LEADING_PREDICTION_PENALTY,
    "lagging_prediction_penalty": LAGGING_PREDICTION_PENALTY,
}
model_configuration_path = MODEL_OUTPUT_DIR / MODEL_CONFIGURATION_FILENAME
model_configuration_path.write_text(json.dumps(model_configuration, indent=4))

display(
    comparison_summary[
        [
            "model",
            "window_hours",
            "mae",
            "rmse",
            "r2",
            "asymmetric_penalized_mae",
            "leading_prediction_rate",
            "lagging_prediction_rate",
        ]
    ]
)

print(
    "Baseline lagging prediction rate: "
    f"{baseline_metrics['lagging_prediction_rate']:.6%}"
)
print(
    "Asymmetric lagging prediction rate: "
    f"{asymmetric_metrics['lagging_prediction_rate']:.6%}"
)
print(f"Saved model configuration to {model_configuration_path}")
print(f"Saved all 4-hour reports and plots under {MODEL_OUTPUT_DIR}")